In [317]:
from datetime import datetime
from dateutil.relativedelta import relativedelta;
import yfinance  as yf
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np


#CL

# Fecthign data
file_name="TSLA_10y.csv";
if(os.path.exists(file_name)):
    stock_data = pd.read_csv(file_name,index_col=0, parse_dates=True);
else:
    ticker = ["TSLA"]
    end_date = datetime.now()
    start_date = end_date -  relativedelta(years=10)
    # Load apple stock data for the last 10 years
    stock_data = yf.download(ticker, start=start_date, end=end_date) 
    stock_data.columns = stock_data.columns.droplevel(1)

# -----------------------------------------------------------------------------

# Calcualte percentage change for each row
stock_data['daily_pct_change'] = stock_data['Close'].pct_change() * 100;

# Dropping NAN data rows.
stock_data = stock_data.dropna()


[*********************100%***********************]  1 of 1 completed


### Feature Selection

Here we are deciding on a bunch of feature columns that we will be using to make predictions.


### Feature 1: volume_ratio_20_40

**Definition**

    average daily volume over the last 20 days
    ----------------------------------------------  
    average daily volume over the 40 days before that

Reading it:
- 1.0 -> recent trading activity matches the baseline
- 1.5 -> recent days are 50% busier than normal
- 0.6  -> recent days are quieter than normal

Needs 60 days of history in total (40 baseline + 20 recent).

**Why I chose it**

A price move backed by heavy volume means a lot of people are acting on it. So volume is my confirmation signal — it tells me whether the
20-day price move has real weight behind it.

**Why a 40-day baseline rather than 20**

More days averaged means a more stable estimate of "normal" volume. Worth it for the stability.

**What it doesn't capture**

The shape of the change. A single volume spike and a steady climb over 20 days produce the same value. It tells me the recent level is elevated, not how it got there.

**A known limitation of the hypothesis itself**

My actual belief is that volume matters *conditional on* price movement — "a big move **with** volume behind it." That's an interaction between two features. A linear model treats each feature independently, so it will pick up volume's standalone effect but not the interaction. I'm accepting that for now an interaction term (price_change x volume_ratio) would be the fix, but I'm keeping the feature count low to limit overfitting.

In [318]:
stock_data['vol_avg_recent_20d']   = stock_data['Volume'].rolling(20).mean()
stock_data['vol_avg_baseline_40d'] = stock_data['Volume'].rolling(40).mean().shift(20)
stock_data['volume_ratio_20_40']   = stock_data['vol_avg_recent_20d'] / stock_data['vol_avg_baseline_40d']

### Feature 2: price_change_20d

**Definition**

    (close today - close 20 days ago) / close 20 days ago  x  100


**Why I chose it**

This is the core of the hypothesis. If a stock has been trending over 20 days, I'm
testing whether that trend continues into the next 5 days.

**Why point-to-point rather than averaging the daily changes**

My first instinct was to average the 20 daily percentage changes. That has a built-in
upward bias.

A price that goes 100 -> 110 -> 100 -> 110 and ends exactly where it started produces
daily changes of +10%, -9.09%, +10%, -9.09%... which average to **+0.455%**. It reports
an uptrend where nothing happened. The cause is that percentage changes are asymmetric:
going up 10% and coming back down is only -9.09%, so oscillation leaves a fake positive
residue.

**What it doesn't capture**

The *path*. A steady climb and a flat month ending in one huge jump produce the same
number. That's what feature 3 is for.

---

### Feature 3: positive_days_20d

**Definition**

Count of days in the last 20 where the daily return was positive. Range 0 to 20.

**Why I chose it**

Feature 2 tells me the net move but nothing about how it happened. A +5% month could be
17 red days rescued by three enormous green ones — or a steady grind upward. Those look
identical to `price_change_20d` and feel completely different as a trend.

This feature measures **consistency**: how often the stock actually went up, ignoring size
entirely. Together the two describe both the *magnitude* and the *reliability* of the move.

**The multicollinearity question**

Measured on the actual data:

| pair | correlation |
|---|---|
| price_change_20d vs positive_days_20d | **0.736** |

Correlated features are a problem because the model struggles to attribute credit between
them. Many different weight splits fit the data almost equally well, so it breaks the tie
using noise rather than signal — producing weights that swing between samples and can't be
read as "this feature matters more than that one".

In [319]:
# Feature 2
stock_data['20_day_price_change'] = stock_data['Close'].pct_change(periods=20)*100

# Feature 3
stock_data['positive_days_20d'] = (stock_data['daily_pct_change'] > 0).rolling(20).sum()



### Feature 4: avg_gap_20d

**Definition**

    daily_gap = (today's Open - yesterday's Close) / yesterday's Close x 100
    avg_gap_20d = mean of the last 20 daily_gap values
    
**Why I chose it**

Markets are shut overnight, so news, earnings and overseas moves accumulate and the price
can open somewhere quite different from where it closed. That overnight jump is a distinct
kind of pressure from intraday drift — it's where information arrives all at once.

If a 20-day trend is genuine, I'd expect it to show up in how the stock keeps *opening*,
not just how it closes.

**Why the mean and not the median**

The mean is outlier-sensitive. I tested this: dropping one -8% earnings gap into an
otherwise mild window flipped the mean from **+0.350 to -0.048** — a sign flip caused by a
single day. The median stayed at +0.245, completely unmoved.

So the median is the robust choice, and I nearly used it. I didn't, because the median
throws away exactly the events I care about. Two windows can have identical medians while
one contains an 8% overnight surge — real news, real information — and the median simply
cannot see it.

There's no measure that keeps magnitude *and* resists outliers, because those are the same
property. Sensitivity to large values is what makes something outlier-vulnerable.

The question is really: **is a big gap signal or noise?** My hypothesis says signal — an
8% earnings pop is the strongest possible version of "overnight pressure supporting a
trend". So I kept the mean, and added a count feature to cover its blind spot.

**What it doesn't capture**

Consistency. One huge gap and twenty small ones can produce similar averages.
That's what feature 5 is for.

---

### Feature 5: gap_up_count_20d

**Definition**

Count of days in the last 20 where the gap was positive. Range 0 to 20.

**Why I chose it**

It covers feature 4's weakness. `avg_gap_20d` sees magnitude but can be flipped by a
single day; this counts *how often* the stock opened higher and is completely immune to
size. Together, if the mean says "gapping up" but the count says "only 6 of 20 days", I
know one outlier is driving the average.

They cover each other's blind spots — magnitude and consistency.

**Why the NaN handling matters**

`daily_gap` has one NaN (the first row, no previous close). Writing `daily_gap > 0` would
silently turn that NaN into **False** — recording "this was not a gap up" when the truth is
"I don't know". The count would then be built partly on a fabricated observation.

`.where(daily_gap.notna())` keeps the NaN as NaN, so any rolling window containing it
correctly refuses to produce a count.

In this dataset it happens not to change anything, because the volume feature needs 60 days
of history and those early rows get dropped anyway. I fixed it regardless — the dataset
should represent what is known, not paper over a gap. And if I ever removed the volume
feature, this bug would silently activate.

---

### Correlation between all five features

|  | price_chg | pos_days | volume | gap_count | avg_gap |
|---|---|---|---|---|---|
| **price_change_20d** | 1.000 | 0.736 | -0.201 | 0.394 | 0.646 |
| **positive_days_20d** | 0.736 | 1.000 | -0.122 | 0.333 | 0.415 |
| **volume_rate_of_change** | -0.201 | -0.122 | 1.000 | -0.010 | -0.251 |
| **gap_up_count_20d** | 0.394 | 0.333 | -0.010 | 1.000 | 0.629 |
| **avg_gap_20d** | 0.646 | 0.415 | -0.251 | 0.629 | 1.000 |

**What this says**

- **Volume is essentially independent of everything** (-0.25 to -0.01). It's carrying its
  own information, which is exactly what I wanted from it.
- The **strongest pair is price_change / positive_days at 0.736** — expected, they're two
  views of the same 20-day move.
- **avg_gap correlates 0.646 with price_change and 0.629 with gap_up_count** — moderate.
  Overnight moves are part of the overall move, so some overlap is unavoidable.
- **gap_up_count vs positive_days is only 0.333** — usefully low. Overnight direction and
  full-day direction are genuinely different things; a stock can gap up and then fall all
  day.

**Proper VIF (not just pairwise)**

With five features, pairwise correlation understates the problem, because a feature can be
predicted by a *combination* of the others even when no single pairing looks high. The
correct VIF regresses each feature against all the rest.

| feature | VIF | SE multiplier |
|---|---|---|
| price_change_20d | 3.19 | 1.79x |
| positive_days_20d | 2.27 | 1.51x |
| volume_rate_of_change | 1.11 | 1.05x |
| gap_up_count_20d | 1.76 | 1.33x |
| avg_gap_20d | 2.63 | 1.62x |

Conventional thresholds: under 5 is fine, 5-10 deserves attention, above 10 is a real
problem. **Everything here is under 3.2.** The condition number of the correlation matrix
is 14.9, also comfortably below the usual concern level of 30.

So the weight standard errors are inflated by roughly 1.3x to 1.8x compared to perfectly
independent features. Real, but well within acceptable range. I'll still train with and
without Ridge and compare weight stability across walk-forward folds, since that gives me
evidence rather than an assertion.

# Feature 4 avg_gap_20_days

stock_data['daily_gap'] = ((stock_data['Open'] - stock_data['Close'].shift(1))
                           / stock_data['Close'].shift(1)) * 100



stock_data['avg_gap_20d'] = stock_data['daily_gap'].rolling(20).mean()


# Feature 5 gap_up_count_20d


is_gap_up = (stock_data['daily_gap'] > 0).where(stock_data['daily_gap'].notna())
stock_data['gap_up_count_20d'] = is_gap_up.rolling(20).sum()

corr = stock_data[['20_day_price_change', 'positive_days_20d',
                   'volume_ratio_20_40', 'gap_up_count_20d',
                   'avg_gap_20d']].corr()

corr.round(3).style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

In [267]:

# Quick inspection of the feature vectors


features_col = ['20_day_price_change', 'positive_days_20d',
                   'volume_ratio_20_40', 'gap_up_count_20d',
                   'avg_gap_20d'];

# Now buidling the target column
# label positive 1 negatove 0

future_close = stock_data['Close'].shift(-5)
stock_data['future_direction_5d'] = (future_close - stock_data['Close'] > 0).where(future_close.notna())


# Adding a new column tatget_2 for price change 

stock_data['future_return_5d'] = ((future_close - stock_data['Close'])/stock_data['Close']) * 100
print("share positive:", stock_data['future_direction_5d'].mean().round(3))
print(stock_data['future_return_5d'].describe().round(2))



share positive: 0.531
count    2507.00
mean        0.98
std         8.63
min       -43.05
25%        -4.19
50%         0.53
75%         5.30
max        56.48
Name: future_return_5d, dtype: float64


In [268]:
# Preapring data for the model
# First combinign the data
combined_data =  stock_data[features_col+['future_return_5d','future_direction_5d']].dropna();
# Now split

input_data = combined_data[features_col];
y = combined_data['future_return_5d'] # Actual percentage chnage
y_return = combined_data['future_direction_5d'] # for the logistic regression later direction data


print('Do the index match ?', input_data.index.equals(y.index))
print('Do the index match ?', input_data.index.equals(y_return.index))

Do the index match ? True
Do the index match ? True


In [269]:
# Starting from psotion 0 and then taking a step of 5 each time

stepped_data = combined_data.iloc[::5]


targets = stepped_data['future_return_5d']
print("neighbour correlation:", round(targets.autocorr(lag=1), 3))

input_stepped_data = input_data.iloc[::5]
y_stepped = y.iloc[::5] 
y_return_stepped = y_return.iloc[::5] 



neighbour correlation: 0.117


### Stepping every 5 days: why the row count drops from 2449 to 490

**The problem**

My target is "what happened over the next 5 days". With one row per trading day,
consecutive rows describe almost the same stretch of time:

    row 1  ->  target covers days  2, 3, 4, 5, 6
    row 2  ->  target covers days     3, 4, 5, 6, 7
    row 3  ->  target covers days        4, 5, 6, 7, 8

Rows 1 and 2 **share 4 of their 5 days**. They are not two separate observations —
they are largely the same fact written down twice.

**Why that breaks the statistics**

The standard error formula is:

    SE = s / sqrt(n)

That formula assumes every row is an independent piece of evidence. It has no way to
know that rows 1 and 2 are mostly the same thing. It just counts rows.

So if I hand it 2449 rows containing only ~490 rows' worth of real information, it
believes it has five times more evidence than it does — and reports far more confidence
than it has earned.

Concretely, with 5-fold overlap:

    what the formula computes  =  s / sqrt(2449)  =  s / 49.5
    what is actually true      =  s / sqrt(490)   =  s / 22.1

    ratio = 49.5 / 22.1 = 2.24 = sqrt(5)

**The standard error comes out about 2.24x too small.** Confidence intervals are 2.24x
too narrow, and p-values are correspondingly too small.


**The fix**

    input_stepped_data = input_data.iloc[::5]

Keep every 5th row: rows 0, 5, 10, 15... Each target now begins exactly where the previous
one ended, so no two share a day.

Rows: **2449 -> 490**

---

### Neighbour correlation, and why -0.051 is the number that matters

**What it measures**

Neighbour correlation (autocorrelation at lag 1) asks:

> *Does one row tell me anything about the next row?*

It takes each column, lines it up against itself shifted by one position, and computes
the correlation. Near zero means each row is fresh information. High means rows are
echoing each other.

This is the direct test of whether the stepping worked.

**My result**

    future_return_5d neighbour correlation:  -0.051

Before stepping this was around **+0.80**. Now it is essentially zero.

That is exactly what I wanted. Consecutive targets no longer share any days, so knowing
one week's return tells me nothing about the next. **The rows are now genuinely
independent observations**, which is precisely what the standard error formula assumes.

---

### What this costs

Going from 2449 rows to 490 feels like discarding 80% of the data. It isn't — I'm
discarding 80% of the *rows* while keeping essentially all of the *information*, since the
dropped rows were near-duplicates. The simulation above confirms this: real precision
barely changed.

What I actually give up is false confidence. My final confidence intervals will be roughly
2.2x wider than the overlapping version would have shown. Same evidence, honestly reported.

In [270]:
# Trainign the model

# Addding a new feature for the bias  paramter

print(input_stepped_data.shape)
print(y_return_stepped.shape)

if 'intercept' not in input_stepped_data.columns:
    input_stepped_data.insert(0,'intercept',1)
print(input_stepped_data.shape)
print(input_stepped_data.head(3))        

(490, 5)
(490,)
(490, 6)
Price       intercept  20_day_price_change  positive_days_20d  \
Date                                                            
2016-11-09          1            -5.682100                8.0   
2016-11-16          1            -9.643352                9.0   
2016-11-23          1            -4.499604               10.0   

Price       volume_ratio_20_40  gap_up_count_20d  avg_gap_20d  
Date                                                           
2016-11-09            1.574624              13.0     0.244907  
2016-11-16            1.447296              11.0     0.164765  
2016-11-23            1.473455              12.0     0.282053  


### Adding an intercept column

My prediction rule so far is:

    prediction = w1*f1 + w2*f2 + w3*f3 + w4*f4 + w5*f5

There's a problem with that. If every feature happened to be zero, every term becomes
zero and the prediction is forced to be exactly 0.

But that's not right. AAPL drifted upward by about +0.58% per 5 days over my sample.
Even a week where the features tell me nothing should probably predict something
slightly positive. My rule can't express that — it's chained to zero.

**The fix** is one extra number that isn't attached to any feature:

    prediction = b + w1*f1 + w2*f2 + ...

That `b` is the **intercept**. It's the model's baseline — what it predicts when the
clues say nothing.

**The trick to compute it:** instead of treating `b` as a special case in the maths,
I add a column to X where every value is **1**. Then `b` is just another weight:

    b*1 + w1*f1 + w2*f2 + ...

Since the column is always 1, `b` passes through unchanged on every row — so it
contributes the same baseline amount to every prediction. Which is exactly what a
baseline should do.

Now the formula has no special cases. It's just "multiply each column by its weight
and add them up", and all six weights work identically.

In [271]:
w = np.linalg.inv(input_stepped_data.T @ input_stepped_data) @ input_stepped_data.T @ y_stepped
w = np.asarray(w)


predictions = input_stepped_data @ w
# print(predictions)
residuals = predictions - y_stepped
# print(residuals)

ss_residual = (residuals ** 2).sum(); # error the model leaves behind
ss_total    = ((y_stepped - y_stepped.mean()) ** 2).sum()   # error if you just predicted the mean

r_squared = 1 - ss_residual/ss_total
print("R² =", round(r_squared, 4))

R² = 0.0335


### Evaluating the model: R²

**What I did to get here**

I solved for the weights using the closed-form normal equations:

    w = (X.T @ X)^-1 @ X.T @ y

That gives the exact set of weights that minimises squared error — no iterating needed,
because a linear model with MSE loss has an exact solution.

But having weights doesn't tell me whether the model is any *good*. I needed a way to
judge it.

---

**The question R² answers**

> Is my model actually better than a lazy guess?

The lazy guess is the dumbest possible model: **ignore every feature and always predict
the average return.** Every week, just say "+0.59%". It looks at nothing.

That's my baseline. Any real model has to beat it or it's worthless.

**How I measure it**

Two totals:

- **`ss_total`** — for every week, how far the real return was from the average, squared,
  all added up. This is the error you're stuck with when you have no information.
- **`ss_residual`** — the same thing using my model's actual predictions. This is the
  error I'm *still* left with after using all five features.

Then:

    R² = 1 - (ss_residual / ss_total)

Reading the fraction: if it's 1.0, my model's error is identical to the lazy guess and I
gained nothing (R² = 0). If it's 0.95, I removed 5% of the error (R² = 0.05).

**So R² is literally "what fraction of the error did I remove by using the features?"**

---

**My result: R² = 0.0335**

My five features explain about **3.35%** of the variation in TSLA's 5-day returns. The
other **96.65% is unexplained pattern** my features can't touch.

*(One methodological note for honesty: I originally ran this on AAPL, which gave 0.0082.
I then tried TSLA. That's a choice made after seeing a result, so I'm declaring it rather
than quietly presenting TSLA as my only run.)*


**Why this number is not trustworthy yet**

** The model was fitted on this exact data.** It has seen every week it's being scored on.
That's marking your own exam. Whatever R² comes out is inflated — it's a **ceiling**, not a
result.

---

**What comes next: walk-forward validation**

Everything above trained on all 490 rows at once. That's the look-ahead problem — the model
knew the future while learning.

The honest test is **walk-forward**: train only on the past, predict a period the model has
never seen, then roll forward and repeat.

    train 2016-2019  ->  test 2020
    train 2016-2020  ->  test 2021
    train 2016-2021  ->  test 2022
    ...

That produces out-of-sample predictions, and an R² I can actually believe. I expect it to
be **lower than 0.0335**

## Walk-forward validation

---

### Why I can't use the standard train/test split

The usual ML approach is to shuffle the data and take an 80/20 split. **For time series
that's catastrophic.**

Shuffle, and my training set ends up containing days from *after* my test days. The model
learns from the future. The backtest looks brilliant and the strategy makes nothing when
traded live, because tomorrow's data doesn't exist yet in the real world.

This is **look-ahead bias**, and it's the single most common way backtests lie.

### But a single chronological split isn't enough either

Training on 2016-2023 and testing on 2024-2026 would be perfectly valid — no future leaks
in. So why not just do that?

Three reasons:

**1. Too little test data.** One test period gives me maybe 100 stepped rows. The
confidence interval on that would be enormous.

**2. I'd be hostage to one period.** I'd learn how the strategy did in *2024-2026*
specifically. If that happened to be a calm bull market, or a crash, or a sideways grind,
my entire verdict would rest on one accident of timing.

**3. It can't tell me whether the edge is stable.** This is the one that matters most here.


**A single split cannot separate "real edge" from "one lucky regime". Walk-forward can** —
because I get a separate score for each period and can look at the sequence.

---

### The approach: expanding window

Walk-forward isn't a different philosophy from the chronological split. It's the same split
done repeatedly, rolling forward through time.

Each fold trains on everything up to a point, then tests on the year that follows:

    train 2016-2021  ->  test 2022
    train 2016-2022  ->  test 2023
    train 2016-2023  ->  test 2024
    train 2016-2024  ->  test 2025
    train 2016-2025  ->  test 2026

Five folds. Every one of them is a valid chronological split — the model never sees data
from after its test period.

---

### What I'm recording per fold

For each fold I'll record:

- **R² on the test set** — did it beat just predicting the average?
- **the weights** — do they stay stable across folds, or swing wildly?
- **row counts** — a test fold with too few rows produces a meaningless R²

The weight stability check is where the multicollinearity question finally gets answered
with evidence rather than assertion. If the weights swing between folds without Ridge and
settle with it, I'll have demonstrated the effect on real data.

### Standardising, and the leakage trap inside it

My features are on wildly different scales — `positive_days_20d` runs 0-20 while
`avg_gap_20d` runs around ±0.5. Weights aren't comparable until I fix that, so I'll
standardise (subtract the mean, divide by the standard deviation).

**But standardising needs a mean and a standard deviation, and those are numbers computed
from data.** If I compute them across the whole dataset, my 2022 training fold gets scaled
using statistics that include 2025. That's leakage — subtler than a bad feature, because
it doesn't look like a feature problem at all.

So the scaling happens **inside each fold**, using that fold's training statistics only:

---
**Each fold is executed and discussed in its own cell below.**

### Fold 1: train 2016–2021, test 2022

**Result: R² = −0.0731**

A **negative** R². This is worse than the lazy baseline — the model does a *poorer* job on
2022 than simply predicting the training-period average every single week.

Out-of-sample R² can go negative (in-sample it can't), and this is what it means: the
features actively hurt rather than helped.

For contrast, the in-sample R² on all the data was +0.0335. So the honest number is not just
smaller — it has crossed to the wrong side of zero.

---

**The weights (features are standardised, so these are directly comparable)**

| feature | weight |
|---|---|
| intercept | +1.6630 |
| **avg_gap_20d** | **+2.4449** |
| **20_day_price_change** | **−2.0321** |
| gap_up_count_20d | +0.7854 |
| volume_ratio_20_40 | +0.3845 |
| positive_days_20d | −0.2914 |

Because the features are standardised, a weight now reads directly as: *"when this feature
moves by one typical amount, the predicted 5-day return shifts by this many percentage
points."* No correction for scale needed.

---

**Observation 1: two weights are implausibly large, and they oppose each other**

`avg_gap_20d` at **+2.44** and `20_day_price_change` at **−2.03** dominate everything else.

A weight of 2.44 claims that one typical move in the gap feature shifts the predicted return
by 2.44 percentage points. My 5-day returns have a standard deviation of roughly 7–8 points,
so that's **about a third of the target's entire spread, from one feature.**

**That cannot be true, and here's why.** R² and correlation are linked: R² = correlation².
My in-sample R² of 0.0335 means the model's predictions correlate with reality at only
√0.0335 ≈ **0.18**. If a single feature genuinely moved the target that much, the model
would track reality far better than 0.18.

So the weight is **inflated**, not powerful.

**Observation 2: the two big weights largely cancel**

`avg_gap_20d` and `20_day_price_change` correlate at about 0.65 — they tend to be high
together. So a large positive weight on one and a large negative weight on the other means
their contributions **mostly annihilate**, leaving a small net prediction.

This is the textbook multicollinearity signature. The model is not saying "gaps push
returns up and momentum pushes them down." It is saying **"I cannot separate these two, so
I'll assign large opposing numbers that happen to fit this training set."**

I flagged exactly this risk when I chose to keep correlated features. Here it is on real
data.

**Observation 3: that cancellation is why R² went negative**

Large opposing weights fit the training data well because the cancellation is tuned to
*that* data's noise. On 2022 — which the model never saw — the two features don't line up
the same way, the cancellation misfires, and the errors come out worse than just guessing
the average.

**Overfitting, caused by multicollinearity, producing a negative out-of-sample R².** The
chain behaved exactly as the theory predicted.

**Observation 4: the smaller weights are residue, not findings**

`positive_days_20d` at −0.29 is counterintuitive — it says more up days predict *lower*
returns. But it's the smallest weight in the model, roughly 4% of the target's spread, and
it sits downstream of two features that are busy cancelling each other out. Whatever is left
over lands on the remaining features.

There *is* a real financial story it could be telling (a stock that has risen consistently
for 20 days is stretched and due a pullback — mean reversion is a well-documented effect).
**But I don't have the evidence to claim that.** At this magnitude, with this much noise and
only 259 training rows, the sign could easily flip on different data.

I said in the feature section that I wouldn't make strong claims about individual weights
given the multicollinearity. This is that situation.

---

**Conclusions from fold 1**

1. Out-of-sample, the model performs **worse than a lazy baseline** on 2022.
2. The weights are **inflated and opposing** — unstable rather than informative.
3. The multicollinearity I identified during feature selection is **visibly damaging the
   model**, not just a theoretical concern.

**What this does NOT establish yet.** One fold is one number, and 2022 was a severe bear
market for TSLA — an unusual year that may not represent the strategy's general behaviour.
I need the remaining four folds before concluding anything.

**What to watch across the remaining folds:**

- **does the sign of each weight stay consistent?** If `avg_gap` is +2.44 here and negative
  next fold, the instability is confirmed beyond doubt.
- **do the large weights persist**, or was this fold unusual?
- **is R² negative everywhere**, or only in 2022?

**And the test this sets up:** if the weights shrink and stabilise under **Ridge**, and R²
improves, I will have demonstrated the whole multicollinearity → instability → overfitting
chain on real data, and shown the fix working — rather than just asserting the theory.

In [342]:
def run_fold(test_year):
    train = stepped_data[stepped_data.index.year <  test_year]
    test  = stepped_data[stepped_data.index.year == test_year]

    X_train = train[features_col]
    Y_train = train['future_return_5d']
    X_test  = test[features_col]
    Y_test  = test['future_return_5d']

    # standardise using TRAIN statistics only
    train_mean = X_train.mean()
    train_std  = X_train.std()
    X_train_s = ((X_train - train_mean) / train_std).copy()
    X_test_s  = ((X_test  - train_mean) / train_std).copy()

    X_train_s.insert(0, 'intercept', 1.0)
    X_test_s.insert(0, 'intercept', 1.0)

    # fit on train only
    w = np.linalg.inv(X_train_s.T @ X_train_s) @ X_train_s.T @ Y_train
    w = np.asarray(w)

    # predict on test
    predictions = X_test_s @ w
    ss_res = ((Y_test - predictions) ** 2).sum()
    ss_tot = ((Y_test - Y_train.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot

    print(f"--- test year {test_year} ---")
    print(f"train {len(train)} rows ({train.index.min().date()} to {train.index.max().date()})")
    print(f"test  {len(test)} rows ({test.index.min().date()} to {test.index.max().date()})")
    print(f"R² = {r2:.4f}")
    for col, weight in zip(X_train_s.columns, w):
        print(f"   {col:25s} {weight:+.4f}")
    print()

    return {'year': test_year, 'r2': r2,
            **dict(zip(X_train_s.columns, w))}

fold1 = run_fold(2022)
fold2 = run_fold(2023)
fold3 = run_fold(2024)
fold4 = run_fold(2025)
fold5 = run_fold(2026)

--- test year 2022 ---
train 259 rows (2016-11-09 to 2021-12-27)
test  51 rows (2022-01-03 to 2022-12-30)
R² = -0.0731
   intercept                 +1.6630
   20_day_price_change       -2.0321
   positive_days_20d         -0.2914
   volume_ratio_20_40        +0.3845
   gap_up_count_20d          +0.7854
   avg_gap_20d               +2.4449

--- test year 2023 ---
train 310 rows (2016-11-09 to 2022-12-30)
test  50 rows (2023-01-09 to 2023-12-29)
R² = -0.0048
   intercept                 +1.0660
   20_day_price_change       -0.8359
   positive_days_20d         -0.3962
   volume_ratio_20_40        +0.2503
   gap_up_count_20d          +1.4089
   avg_gap_20d               +1.0753

--- test year 2024 ---
train 360 rows (2016-11-09 to 2023-12-29)
test  50 rows (2024-01-08 to 2024-12-27)
R² = -0.0026
   intercept                 +1.1522
   20_day_price_change       -0.6554
   positive_days_20d         -0.4617
   volume_ratio_20_40        +0.6026
   gap_up_count_20d          +1.3869
   avg_gap_2

### Walk-forward results: all five folds

**R² by fold**

| test year | test rows | R² |
|---|---|---|
| 2022 | 51 | −0.0731 |
| 2023 | 50 | −0.0048 |
| 2024 | 50 | −0.0026 |
| 2025 | 50 | −0.0406 |
| 2026 | **30** | **+0.0976** |

| | |
|---|---|
| simple average | **−0.0047** |
| weighted by test rows | −0.0139 |
| excluding 2026 | **−0.0303** |
| negative folds | **4 out of 5** |

---

### The headline result

**Four of the five folds are negative.** In most years the model performs *worse* than
simply predicting the training-period average every week. The features aren't just failing
to help — they're actively hurting.

**The one positive fold is the least trustworthy.** 2026 has only 30 rows because the data
ends in August, making it by far the noisiest estimate of the five. Drop it and the average
falls from −0.005 to **−0.030**.

**The two numbers that summarise this whole exercise:**

    in-sample R²      = +0.0335
    walk-forward R²   = -0.0047

**+0.034 when the model marks its own exam. −0.005 when it doesn't.**

That gap is the entire reason walk-forward validation exists. Had I stopped at the in-sample
figure, I would have concluded there was a modest edge here. There isn't.

---

### Weight stability across folds

| feature | 2022 | 2023 | 2024 | 2025 | 2026 | sign stable? | range |
|---|---|---|---|---|---|---|---|
| price_change_20d | −2.03 | −0.84 | −0.66 | −0.53 | −0.21 | yes | 1.82 |
| positive_days_20d | −0.29 | −0.40 | −0.46 | −0.87 | −0.86 | yes | 0.58 |
| volume_ratio_20_40 | +0.38 | +0.25 | +0.60 | +0.41 | +0.46 | yes | 0.35 |
| gap_up_count_20d | +0.79 | +1.41 | +1.39 | +1.17 | +1.24 | yes | 0.62 |
| avg_gap_20d | +2.44 | +1.08 | +0.84 | +1.07 | +0.65 | yes | 1.80 |

**Every sign held across all five folds.** No feature flipped direction. That looks like
consistency, and it's better than I expected after fold 1.

**But I should not over-read it.** The folds share most of their training data — fold 1
trains on 259 rows, fold 5 on 460, and **those first 259 rows appear in every single fold**.
The training sets are 55–100% identical. Weights *ought* to be similar. Sign stability here
is much weaker evidence than it looks, and I'd be fooling myself to treat it as
confirmation of a real relationship.

---

### The most informative pattern: the big weights are shrinking

    price_change_20d:  -2.03  ->  -0.84  ->  -0.66  ->  -0.53  ->  -0.21
    avg_gap_20d:       +2.44  ->  +1.08  ->  +0.84  ->  +1.07  ->  +0.65

These are the two large opposing weights I identified in fold 1 as the multicollinearity
signature. **Both are marching toward zero as the training set grows.**

That's the theory playing out on real data: more rows means less need for extreme
cancellation to fit the noise. It's the "more data reduces variance" effect, which is one
of the three fixes for multicollinearity (alongside Ridge and dropping a feature).

**And note where `price_change_20d` is heading: −0.21.** The feature at the very centre of
my hypothesis is converging on approximately nothing.

Meanwhile `positive_days_20d` moves the other way (−0.29 → −0.86), growing as the momentum
feature shrinks. Those two correlate at 0.73, so this looks like credit shifting between
them as the data changes — which is the multicollinearity again, not a discovery about
consistency mattering more over time.

---

### What this establishes

**My hypothesis is not supported.**

> *"A stock trending over 20 days tends to continue over the next 5, provided the move is
> backed by genuine participation."*

Out-of-sample, a model built on that idea is **worse than guessing the average**.

This isn't a failure of the work — it's the work doing its job. Momentum plus volume on a
single large-cap stock is one of the most obvious ideas in trading. If it reliably predicted
next week's returns it would have been arbitraged away long ago. **Finding nothing was
always the most likely honest outcome.**

I also note that I switched from AAPL (in-sample R² 0.0082, below the pure-noise floor of
~0.010) to TSLA after seeing that result. TSLA's higher in-sample figure did not survive
walk-forward either.

---